---
#### *Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026*
---

# 2.1 - Evaluation Refinement
---
## 💬 Product Team Check-in

> **Sam:** "Hey — the beta testers in ML Engineering have been active. Two pieces of feedback worth acting on. First, the tone feels a bit robotic. Users said it feels like they're reading documentation rather than talking to a helpful assistant."
>
> "Second thing — users are asking non-MLflow questions and the agent is just... answering them. Someone asked for a pasta recipe and it gave them one."
>
> "One more thing - Is there a way we can identify examples in the MLflow server and add to your evaluation criteria?"

**Objective:** Translate feedback into evaluations and scorers that can be used to refine the agent. Refine aspects of the agent to meet updated evaluation criteria.

In this notebook:

- Add some ops upgrades to our pipeline
- Introduce Evaluation Datasets
- Introduce Prompt Registry + Prompt Aliases
- Add Guidelines Scorers
- Make incremental improvements to system prompt

---

## 0 - Connect to MLflow (Migrate to Pydantic Settings)

In [ ]:
uv add pydantic-settings
uv sync

In [ ]:
from functools import cached_property
from pathlib import Path
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ENV_FILE = Path.cwd().parent.parent / ".env"  # adjust .parent count to match your nesting

class AgentConfig(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=ENV_FILE,
        env_file_encoding="utf-8",
        extra="ignore",  # don't choke on unrelated vars in .env
    )

    # Secrets / endpoints (from .env)
    gemini_api_key: SecretStr
    gemini_openai_base_url: str

    # MLflow (from .env, with fallbacks)
    mlflow_tracking_uri: str
    experiment_name: str = Field(default="mlflow-agent-1", alias="EXPERIMENT_1_NAME")

    # Model knobs (hardcoded defaults, overridable via env if you want)
    model: str = "gemini-2.5-flash-lite"
    temperature: float = 0.2
    max_tokens: int = 512
    ## Adding judge model to config
    judge_model: str = "gemini:/gemini-3.1-flash-lite-preview"

    # Prompt
    prompt_name: str = "mlflow-agent-system"
    system_prompt: str = "You are an MLflow assistant."

    ##DATASET
    eval_dataset_name: str = "mlflow-agent-eval"

    @cached_property
    def prompt_alias(self) -> str:
        return f"prompts:/{self.prompt_name}@prod"

cfg = AgentConfig()

In [ ]:
from openai import OpenAI
import mlflow

openai_client = OpenAI(
    api_key=cfg.gemini_api_key.get_secret_value(),
    base_url=cfg.gemini_openai_base_url,
)

mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.experiment_name)
mlflow.openai.autolog()

In [ ]:
def mlflow_agent(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=cfg.model,
        messages=[
            {"role": "system", "content": cfg.system_prompt},
            {"role": "user",   "content": question},
        ],
        temperature=cfg.temperature,
        max_completion_tokens=cfg.max_tokens,
    )
    return response.choices[0].message.content

## 1 - Operational Upgrades

### Change Judge Model

Using the same LLM as both agent and judge creates correlated blind spots — the judge shares the agent's biases and failure modes, so it's more likely to rate flawed outputs as correct.

Common practice to choose a more 'intelligent' or 'capable' model to evaluate responses.

![](../../assets/aa_intelligence_index.png)

### Evaluation Datasets

#### The Problem with Inline Datasets

**Scaling issues:**
- Adding a new example means editing notebook code and re-running
- Creating via separate .py file can get long
- No way to tell who added which example, or when
- If two people add the same question, you get duplicates
- Deleting a bad example means finding it in a long list and hoping you don't break the format
- The dataset lives in notebook state — restart the kernel and it's gone until you re-run

Let's fix all of this by creating a **persistent evaluation dataset** with `mlflow.genai.datasets`. 


Evaluation datasets have the following features:


- **Persistence** — examples stored in a database, not scattered across notebook cells
- **CRUD operations** — add, update, or remove examples without re-running everything
- **Deduplication** — same inputs automatically merge instead of creating duplicates
- **Provenance** — know where each example came from (hand-written, from traces, from docs)
- **Tagging** — organize examples by category, priority, or sprint
- **Versioning** — the dataset has a content hash so you know when it changed

`create_dataset()` creates a new named dataset stored in your tracking server's SQL backend. It's attached to one or more experiments and can be tagged for organization.

**Requirements:** Your tracking server must use a SQL backend (SQLite, PostgreSQL, MySQL), which is the norm going forward. The `sqlite:///mlflow.db` we've been using works perfectly.


In [ ]:
from mlflow.genai.datasets import create_dataset

full_eval_data = create_dataset(
    name="mlflow-agent-eval",
    tags={
        "experiment": cfg.experiment_name,
        "domain": "mlflow",
        "status": "active",
    },
)
full_eval_data.has_records()

In [ ]:
type(full_eval_data)

WARNING: [Documentation issue](https://github.com/mlflow/mlflow/issues/22712)

In [ ]:
#Retrieve the dataset later
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name=cfg.eval_dataset_name)
# If multiple datasets have the same name, you can also retrieve by ID:
# retrieved_dataset = get_dataset(dataset_id="your-dataset-id")

### Add evaluation examples from previous notebook

`merge_records()` adds examples to the dataset. It's called "merge" because if you add an example with the same `inputs` as an existing record, it **updates** the existing record rather than creating a duplicate. This is based on an input hash — if the `inputs` dict is identical, it's the same record.


In [ ]:
eval_dataset_v3 = [
    {
        "inputs": {"question": "How do I log a metric in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.log_metric", "key", "value", "step"],
        },
    },
    {
        "inputs": {"question": "How do I set up autologging for a scikit-learn model?"},
        "expectations": {
            "expected_response": "To enable autologging for scikit-learn in MLflow, call mlflow.sklearn.autolog() before your training code. This automatically logs parameters, metrics, and the trained model to your active run.",
        },
    },
    {
        "inputs": {"question": "How do I organize my runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.set_experiment", "experiment", "run"],
        },
    },
    {
        "inputs": {"question": "How do I log a model artifact in MLflow?"},
        "expectations": {
            "expected_response": "You can log a model artifact using mlflow.log_artifact() for individual files, or framework-specific methods like mlflow.sklearn.log_model() which saves the model along with its dependencies and a model signature.",
        },
    },
    {
        "inputs": {"question": "How do I compare runs in MLflow?"},
        "expectations": {
            "expected_facts": ["mlflow.search_runs", "experiment_id", "metrics"],
        },
    },
]
print(f"Eval set size: {len(eval_dataset_v3)} examples")
#MLflow v3.11.1 bug: print(f"Evaluation data set size before merging: {len(full_eval_data.to_df())} records")
full_eval_data.merge_records(eval_dataset_v3)
print(f"Evaluation data set size after merging: {len(full_eval_data.to_df())} records")

### Running eval in CI

```python
# This is all your CI script needs — the dataset is already populated
dataset = get_dataset(name="mlflow-agent-eval")
results = mlflow.genai.evaluate(data=dataset, predict_fn=recipe_agent, scorers=[...])
assert results.metrics["Correctness/percentage"] >= 0.8
```

### Growing from traces

```python
# Pull interesting production traces into your eval set
traces = mlflow.search_traces(
    filter_string="tags.user_feedback = 'negative'",
    max_results=20,
    return_type="list",
)
dataset.merge_records(traces)
```

In [ ]:
# Add a trace to the mlflow dataset via UI
print(mlflow_agent("How do I search MLflow runs?"))

### Try the 'Add To Dataset' button

In [ ]:
from mlflow.genai.datasets import get_dataset
retrieved_dataset = get_dataset(name=cfg.eval_dataset_name)
eval_df = retrieved_dataset.to_df()
eval_df

In [ ]:
full_eval_data.delete_records([''])

### Add Agent Trace

In [ ]:
@mlflow.trace(name="mlflow_agent", span_type="AGENT")
def mlflow_agent_trace(question: str) -> str:
    response = openai_client.chat.completions.create(
        model=cfg.model,
        messages=[
            {"role": "system", "content": cfg.system_prompt},
            {"role": "user", "content": question},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content

In [ ]:
mlflow_agent_trace("How do I search MLflow runs?")

In [ ]:
# Update Expectations
search_example = [
    {
        "inputs": {"question": "How do I search MLflow runs?"},
        "expectations": {
            "expected_facts": ["mlflow.search_runs"],
        },
    }
]
full_eval_data.merge_records(search_example)

# Problem Validation

## Introducing Guidelines Scorers

There are two ways to evaluate responses against custom rules in plain English. Which one you use depends on whether the guideline is **global** (same rule for every row) or **per-row** (different rules for different questions).

---

### Option A: `Guidelines` — Global rules passed to the scorer

Use this when the same rule applies to every example in your dataset. The scorer itself holds the rule.

**Required parameters:**
- **`name`** — a short identifier for the guideline
- **`guidelines`** — the rule the response must follow

```python
from mlflow.genai.scorers import Guidelines

# Applied to every row in the dataset
Guidelines(
    name="stays_on_topic",
    guidelines="Responses must only address MLflow-related questions.",
)
```

---

### Option B: `ExpectationsGuidelines` — Per-row rules in the dataset

Use this when different questions need different rules. The guidelines live in each row's `expectations.guidelines` list, and the scorer reads them automatically.

```python
from mlflow.genai.scorers import ExpectationsGuidelines

# No name or guidelines on the scorer — it reads from each row
ExpectationsGuidelines()
```

The dataset provides the rules:

```python
{
    "inputs": {"question": "Can you give me a pasta recipe?"},
    "expectations": {
        "guidelines": [
            "Response must not provide non-MLflow content",
            "Response should redirect the user to ask an MLflow question",
        ],
    },
}
```

---

**We'll use `ExpectationsGuidelines` here** since our two validation examples (tone and off-topic) each need their own specific rules.

In [ ]:
validation_dataset_w_guidelines = [
    # Robotic tone — the bare "You are an MLflow assistant" prompt
    # tends to produce dry, documentation-style responses
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
        "expectations": {
            "guidelines": [
                "Response should feel like advice from a knowledgeable colleague, "
                "not a dry enumeration of features or a copy-pasted changelog",
                "Response should explain the practical benefit to the user, "
                "not just list what the feature does",
            ],
        },
    },
    # Off-topic — the bare LLM will happily answer this
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any non-MLflow content",
                "Response should politely redirect the user to ask an MLflow-related question",
            ],
        },
    },
]

In [ ]:
# Evaluate
from mlflow.genai import evaluate
from mlflow.genai.scorers import ExpectationsGuidelines
1
validation_results = evaluate(
    data=validation_dataset_w_guidelines,
    predict_fn=mlflow_agent,
    scorers=[ExpectationsGuidelines(model=cfg.judge_model)],
)

In [ ]:
validation_dataset_w_o_guidelines = [
    # Robotic tone — the bare "You are an MLflow assistant" prompt
    # tends to produce dry, documentation-style responses
    {
        "inputs": {"question": "What is MLflow tracing and why would I use it?"},
    },
    # Off-topic — the bare LLM will happily answer this
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
    },
]

In [ ]:
# Evaluate
from mlflow.genai.scorers import Guidelines

validation_results = evaluate(
    data=validation_dataset_w_o_guidelines,
    predict_fn=mlflow_agent,
    scorers=[Guidelines(
        name="Non-robotic tone",
        guidelines="""Response should feel like advice from a knowledgeable colleague, "
                "not a dry enumeration of features or a copy-pasted changelog",
                "Response should explain the practical benefit to the user, "
                "not just list what the feature does""",
        model=cfg.judge_model),
        Guidelines(
        name="On Topic",
        guidelines="""Response should be related to MLflow context. 
        If question if off topic, response should politely redirect the user to ask an MLflow-related question""",
        model=cfg.judge_model),
        ]
)

## Add specific examples to dataset

In [ ]:
full_eval_data.merge_records(validation_dataset_w_guidelines)
print(f"Number of records: {len(full_eval_data.to_df())}")

# Prompt Registration

---
## The Problem: Prompts as Strings

Our system prompt in its current form:

```python
SYSTEM_PROMPT = """You are an MLflow assistant."""

# After some feedback
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant...Always include code examples and version context."""
```

Questions this process can't answer:
- Which prompt version was running when User #347 got a wrong API recommendation?
- Did changing "be practical" to "include code examples" improve Correctness scores?
- Can we roll back to last week's prompt without a code deploy?

The prompt registry makes all of this much more streamlined.

---
## 2 — Registering the System Prompt

`mlflow.genai.register_prompt()` creates a new prompt (or a new version of an existing one) in the registry. The prompt is stored in the tracking server's database and is immediately available to any notebook, script, or service that can reach the server.

Let's register the original system prompt from the first notebook — the bare-bones version before any of Sam's feedback.

In [ ]:
prompt_v1 = mlflow.genai.register_prompt(
    name=cfg.prompt_name,
    template=(cfg.system_prompt),
    commit_message="Initial system prompt — bare PoC version",
    tags={
        "author": "ai-team",
        "agent": "mlflow-agent",
    },
)

print(f"Prompt: {prompt_v1.name}")
print(f"Version: {prompt_v1.version}")
print(f"Template: {prompt_v1.template[:80]}...")

# Set Prompt Alias

A prompt alias in MLflow is a mutable, named reference (like "production" or "staging") that points to a specific prompt version in the Prompt Registry. To promote a new prompt version to production, assign the "production" alias to the desired version using `mlflow.genai.set_prompt_alias()`. This allows your applications to reference the alias (e.g., prompts:/my_prompt@production) instead of a hard-coded version, enabling seamless updates without redeploying code.

In [ ]:
mlflow.genai.set_prompt_alias(name=cfg.prompt_name, version=1, alias="prod")

In [ ]:
# Update the predict_fn to use the registered prompt + alias
def mlflow_agent(question: str) -> str:
    """MLflow assistant with updated prompt."""
    system_prompt = mlflow.genai.load_prompt(cfg.prompt_alias)
    response = openai_client.chat.completions.create(
        model=cfg.model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
        temperature=cfg.temperature,
        max_completion_tokens=cfg.max_completion_tokens,
    )
    return response.choices[0].message.content

In [ ]:
eval_dataset_v4 = [
    {
        "inputs": {"question": "Can you give me a good pasta recipe?"},
        "expectations": {
            "guidelines": [
                "Response must not provide a pasta recipe or any cooking-related content",
            ],
        },
    },
    {
        "inputs": {"question": "Is there a seahorse emoji?"},
        "expectations": {
            "guidelines": [
                "Response must not answer questions that aren't related to MLflow",
            ],
        },
    },
]
print(f"Eval set size: {len(eval_dataset_v4)} examples")

In [ ]:
# Evaluate
from mlflow.genai.scorers import ExpectationsGuidelines

validation_results = evaluate(
    data=validation_dataset_w_guidelines,
    predict_fn=mlflow_agent,
    scorers=[ExpectationsGuidelines(model=cfg.judge_model)],
)

---

## 3 - Update System Prompt

In [ ]:
SYSTEM_PROMPT_V2 = """You are a helpful MLflow assistant.
Be practical and conversational — like a knowledgeable colleague, not a changelog.
Do not answer questions that aren't related to MLflow."""

In [ ]:
prompt_v2 = mlflow.genai.register_prompt(
    name=cfg.prompt_name,
    template=(SYSTEM_PROMPT_V2),
    commit_message="Added code examples, version context, and conversational tone per Sam's feedback",
    tags={
        "author": "ai-team",
        "agent": "mlflow-agent",
    },
)

print(f"Prompt: {prompt_v2.name}")
print(f"Version: {prompt_v2.version}")
print(f"Template: {prompt_v2.template[:80]}...")

In [ ]:
# Load Prompt
mlflow.genai.load_prompt("prompts:/mlflow-agent-system/2")

In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Completeness
results = mlflow.genai.evaluate(
    data=full_eval_data,
    predict_fn=mlflow_agent,
    scorers=[
        Correctness(model=cfg.judge_model),
        RelevanceToQuery(model=cfg.judge_model),
        Completeness(model=cfg.judge_model),
        ExpectationsGuidelines(model=cfg.judge_model),
    ],
)

# Update to Product Team